### DATA INGESTION 

**INITIALIZE SPARKNLP**

In [ ]:
from sparknlp.base import DocumentAssembler, Finisher
from sparknlp.annotator import Tokenizer, Normalizer, Stemmer, StopWordsCleaner, BertEmbeddings, SentenceEmbeddings, ClassifierDLApproach

from pyspark.ml import Pipeline
from pyspark.ml.feature import CountVectorizer, IDF, NGram, VectorAssembler, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

from pyspark.sql.functions import col, expr, desc, create_map, lit

from itertools import chain

import sparknlp

In [2]:
spark = sparknlp.start(apple_silicon=True)
spark

25/08/30 11:17:28 WARN Utils: Your hostname, m4mba.local resolves to a loopback address: 127.0.0.1, but we couldn't find any external IP address!
25/08/30 11:17:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/jefferyjapheth/.ivy2/cache
The jars for the packages stored in: /Users/jefferyjapheth/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp-silicon_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-461fe1df-a25d-4742-8282-bc55c0478162;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp-silicon_2.12;6.1.2 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central


:: loading settings :: url = jar:file:/Users/jefferyjapheth/miniconda3/envs/sparknlp/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in central
	found com.amazonaws#jmespath-java;1.12.500 in central
	found com.github.universal-automata#liblevenshtein;3.0.0 in central
	found com.google.protobuf#protobuf-java-util;3.0.0-beta-3 in central
	found com.google.protobuf#protobuf-java;3.0.0-beta-3 in central
	found com.google.code.gson#gson;2.3 in central
	found it.unimi.dsi#fastutil;7.0.12 in central
	found org.projectlombok#lombok;1.16.8 in central
	found com.google.cloud#google-cloud-storage;2.20.1 in central
	found com.google.guava#guava;31.1-jre in central
	found com.google.guava#failureaccess;1.0.1 in central
	found com.google.guava#listenablefuture;99

**LOAD CLEAN MCC DATASET**

In [3]:
# Load the cleaned contract dataset
df = spark.read.parquet("../data/processed/mcc_contracts")

print(f"Dataset loaded! Total records: {df.count():,}")
df.printSchema()

Dataset loaded! Total records: 343,849
root
 |-- contract: string (nullable = true)
 |-- description: string (nullable = true)
 |-- agreement_type: string (nullable = true)
 |-- type_score: string (nullable = true)
 |-- data_split: integer (nullable = true)
 |-- label_count: long (nullable = true)
 |-- type_label: string (nullable = true)



### DATA EXPLORATION TO SHOW CLASS DISTRIBUTION/ DATA SPLITS AND SAMPLE RECORDS

In [26]:
# Step 1: Data Exploration and Basic Stats

# Fix type_score data type
df = df.withColumn("type_score", col("type_score").cast("float"))

# # Show class distribution
# print("Class Distribution:")
# df.groupBy("agreement_type", "type_label").count().orderBy(desc("count")).show()

# # Check data splits
# print("Data Split Distribution:")
# df.groupBy("data_split").count().show()

# Sample records to see the text
print("Sample Records:")
df.select("contract", "description", "agreement_type", "type_score").limit(3).show(truncate=False)

Sample Records:
+-------------------------------+---------------------------------------------------------------+--------------+----------+
|contract                       |description                                                    |agreement_type|type_score|
+-------------------------------+---------------------------------------------------------------+--------------+----------+
|b53262vpexv10w24.txt           |form of amended and restated acquisition/capital line of credit|security      |0.99999964|
|apc2015-12x178xkxexhibit101.htm|amendment and maturity extension agreement, december 14, 2015  |security      |0.99999964|
|creditagreement112701.htm      |exhibit 99.2 credit agreement                                  |security      |0.99999905|
+-------------------------------+---------------------------------------------------------------+--------------+----------+



### NLP PIPELINE FOR CLASSICAL ML MODELS

In [ ]:
# Document
document_assembler = DocumentAssembler() \
    .setInputCol("description") \
    .setOutputCol("document")

# Tokenize
tokenizer = Tokenizer() \
    .setInputCols(["document"]) \
    .setOutputCol("tokens")

# Normalize
normalizer = Normalizer() \
    .setInputCols(["tokens"]) \
    .setOutputCol("normalized_tokens") \
    .setLowercase(True)

# Stemmer (better for TF-IDF classification)
stemmer = Stemmer() \
    .setInputCols(["normalized_tokens"]) \
    .setOutputCol("stem_tokens")

# Stopwords
stopwords_cleaner = StopWordsCleaner() \
    .setInputCols(["stem_tokens"]) \
    .setOutputCol("clean_tokens") \
    .setCaseSensitive(False)

# Finisher
finisher = Finisher() \
    .setInputCols(["clean_tokens"]) \
    .setOutputCols(["finished_tokens"]) \
    .setOutputAsArray(True)

pipeline = Pipeline(stages=[
    document_assembler,
    tokenizer,
    normalizer,
    stemmer,
    stopwords_cleaner,
    finisher
])

**FIT AND TRANSFORM RAW DATASET WITH NLP**

In [6]:
nlp_model = pipeline.fit(df)
df_tokens = nlp_model.transform(df)

df_tokens = df_tokens.withColumn(
    "finished_tokens",
    expr("filter(finished_tokens, x -> length(x) > 2)")
)

# Check results
df_tokens.select("contract", "finished_tokens").show(20, truncate=False)


+-------------------------------+-------------------------------------------------------------+
|contract                       |finished_tokens                                              |
+-------------------------------+-------------------------------------------------------------+
|b53262vpexv10w24.txt           |[form, amend, restat, acquisitioncapit, line, credit]        |
|apc2015-12x178xkxexhibit101.htm|[amend, matur, extens, agreem, decemb]                       |
|creditagreement112701.htm      |[exhibit, credit, agreem]                                    |
|f10k123120_ex10z44.htm         |[exhibit, secur, replac, note, date, april]                  |
|dex105.htm                     |[secur, purchas, agreem]                                     |
|dex106.txt                     |[waiver, concern, secur, agreem]                             |
|c83266exv10we.txt              |[amend, credit, agreem]                                      |
|specimen_regswbba.htm          |[specim

### TACKLING CLASS IMBALANCE 

**Apply Under-sampling**
+ Realized the data was highly imbalanced - class imbalance problem 
+ Combining undersampling with class weights to reduce the class imbalance

In [7]:
# Cache tokenized data for performance
df_tokens = df_tokens.cache()
df_tokens.count()  # Trigger caching

# Define target counts for strategic under-sampling
target_counts = {
    'LABEL_1': 50000,  # employment (down from 144k)
    'LABEL_0': 35000,  # security (down from 102k)
    'LABEL_4': 25000,  # purchase&ma (down from 47k)
    'LABEL_3': 20000,  # services&supply (down from 24k)
    'LABEL_5': 15716,  # shareholder (keep all)
    'LABEL_2': 8584    # lease (keep all)
}

print("Applying under-sampling...")
sampled_dfs = []
for label, target_count in target_counts.items():
    label_df = df_tokens.filter(col("type_label") == label)
    current_count = label_df.count()
    
    if current_count > target_count:
        fraction = target_count / current_count
        sampled_df = label_df.sample(withReplacement=False, fraction=fraction, seed=42)
        print(f"{label}: Sampled {target_count:,} from {current_count:,} ({fraction:.3f} fraction)")
    else:
        sampled_df = label_df
        print(f"{label}: Kept all {current_count:,}")
    
    sampled_dfs.append(sampled_df)

# Combine all sampled classes
balanced_df = sampled_dfs[0]
for df_part in sampled_dfs[1:]:
    balanced_df = balanced_df.union(df_part)

print(f"\nTotal records after under-sampling: {balanced_df.count():,}")
print("\nNew class distribution:")
balanced_df.groupBy("agreement_type", "type_label").count().orderBy(desc("count")).show()

Applying under-sampling...
LABEL_1: Sampled 50,000 from 144,489 (0.346 fraction)
LABEL_0: Sampled 35,000 from 102,328 (0.342 fraction)
LABEL_4: Sampled 25,000 from 47,850 (0.522 fraction)
LABEL_3: Sampled 20,000 from 24,882 (0.804 fraction)
LABEL_5: Kept all 15,716
LABEL_2: Kept all 8,584

Total records after under-sampling: 154,255

New class distribution:
+---------------+----------+-----+
| agreement_type|type_label|count|
+---------------+----------+-----+
|     employment|   LABEL_1|49879|
|       security|   LABEL_0|34975|
|    purchase&ma|   LABEL_4|25012|
|services&supply|   LABEL_3|20089|
|    shareholder|   LABEL_5|15716|
|          lease|   LABEL_2| 8584|
+---------------+----------+-----+



#### FEATURE PIPELINE

**N-GRAMS, COUNT VECTORIZER, TF-IDF Feature Extraction**

In [8]:
# Create bigrams
bigram = NGram(n=2, inputCol="finished_tokens", outputCol="bigrams")

# CountVectorizer for unigrams
cv_unigram = CountVectorizer(inputCol="finished_tokens", outputCol="cv_unigram", vocabSize=25000, minDF=2)

# CountVectorizer for bigrams
cv_bigram = CountVectorizer(inputCol="bigrams", outputCol="cv_bigram", vocabSize=20000, minDF=2)

# Combine features (sparse vector concatenation)

assembler = VectorAssembler(inputCols=["cv_unigram", "cv_bigram"], outputCol="raw_features")

# IDF
idf = IDF(inputCol="raw_features", outputCol="tfidf_features")

from pyspark.ml.feature import StringIndexerModel

fixed_labels = ["LABEL_0", "LABEL_1", "LABEL_2", "LABEL_3", "LABEL_4", "LABEL_5"]

label_indexer = StringIndexerModel.from_labels(
    labels=fixed_labels,
    inputCol="type_label",
    outputCol="label_indexed"
)


# Full pipeline
feature_pipeline = Pipeline(
    stages=[    bigram,
                cv_unigram,
                cv_bigram, 
                assembler,
                idf, 
                label_indexer
                ])

# Fit and transform
feature_model = feature_pipeline.fit(balanced_df)
feature_df = feature_model.transform(balanced_df)

feature_df.select("contract", "agreement_type", "tfidf_features", "label_indexed").show(20)

25/08/30 11:17:52 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
25/08/30 11:17:55 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB


+--------------------+--------------+--------------------+-------------+
|            contract|agreement_type|      tfidf_features|label_indexed|
+--------------------+--------------+--------------------+-------------+
|          dex104.htm|    employment|(32174,[1,3,6,18,...|          1.0|
|          dex102.htm|    employment|(32174,[0,2,42,45...|          1.0|
|   c95226exv10w1.htm|    employment|(32174,[1,3,6,18,...|          1.0|
|   d379056dex101.htm|    employment|(32174,[0,77,82,1...|          1.0|
|         dex1022.htm|    employment|(32174,[7,14,19,1...|          1.0|
|  d274705dex1021.htm|    employment|(32174,[6,25,34,6...|          1.0|
|   d372980dex102.htm|    employment|(32174,[5,6,25,26...|          1.0|
|villageedocs06192...|    employment|(32174,[7],[2.586...|          1.0|
|         dex1005.txt|    employment|(32174,[180,644,1...|          1.0|
|f8k120618ex10-1_c...|    employment|(32174,[0,2,22,38...|          1.0|
|exhibitperformanc...|    employment|(32174,[0,6,25

25/08/30 11:17:55 WARN DAGScheduler: Broadcasting large task binary with size 1112.7 KiB


In [28]:
# Cache tokenized data for performance
feature_df = feature_df.cache()  

#confirm custom label mappings
feature_df.groupBy("agreement_type","label_indexed","type_label").count().orderBy(desc("count")).show() # Triggers caching

25/08/30 11:44:38 WARN CacheManager: Asked to cache already cached data.
25/08/30 11:44:39 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB


+---------------+-------------+----------+-----+
| agreement_type|label_indexed|type_label|count|
+---------------+-------------+----------+-----+
|     employment|          1.0|   LABEL_1|49879|
|       security|          0.0|   LABEL_0|34975|
|    purchase&ma|          4.0|   LABEL_4|25012|
|services&supply|          3.0|   LABEL_3|20089|
|    shareholder|          5.0|   LABEL_5|15716|
|          lease|          2.0|   LABEL_2| 8584|
+---------------+-------------+----------+-----+



25/08/30 11:44:39 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB


**Calculate Class Weights**

In [10]:
import builtins 

# Get class counts
class_counts = feature_df.groupBy("label_indexed").count().collect()

# Use Python's built-in sum
total_samples = builtins.sum(row['count'] for row in class_counts)
n_classes = len(class_counts)

class_weights = {}
for row in class_counts:
    label = row['label_indexed']
    count = row['count']
    weight = total_samples / (n_classes * count)
    class_weights[label] = weight

print("Class weights for models:")
for label, weight in class_weights.items():
    print(f"Label {label}: {weight:.3f}")


Class weights for models:
Label 1.0: 0.515
Label 0.0: 0.735
Label 4.0: 1.028
Label 3.0: 1.280
Label 5.0: 1.636
Label 2.0: 2.995


**PREPARE TRAIN/ VAL/ TEST SPLITS FOR MODEL**

In [11]:
# Split based on pre-defined split column
train_df = feature_df.filter(col("data_split").isin([1, 2, 3]))
val_df   = feature_df.filter(col("data_split") == 4)
test_df  = feature_df.filter(col("data_split") == 5)

print("Data splits after balancing and feature extraction:")
print(f"Train: {train_df.count():,}")
print(f"Validation: {val_df.count():,}")
print(f"Test: {test_df.count():,}")

# Inspect class balance
print("\nTrain split class distribution:")
train_df.groupBy("agreement_type").count().orderBy(desc("count")).show(truncate=False)

print("\nValidation split class distribution:")
val_df.groupBy("agreement_type").count().orderBy(desc("count")).show(truncate=False)

print("\nTest split class distribution:")
test_df.groupBy("agreement_type").count().orderBy(desc("count")).show(truncate=False)


Data splits after balancing and feature extraction:
Train: 92,564
Validation: 30,624
Test: 31,067

Train split class distribution:
+---------------+-----+
|agreement_type |count|
+---------------+-----+
|employment     |29946|
|security       |20876|
|purchase&ma    |15088|
|services&supply|12011|
|shareholder    |9505 |
|lease          |5138 |
+---------------+-----+


Validation split class distribution:
+---------------+-----+
|agreement_type |count|
+---------------+-----+
|employment     |9913 |
|security       |6954 |
|purchase&ma    |4950 |
|services&supply|4041 |
|shareholder    |3064 |
|lease          |1702 |
+---------------+-----+


Test split class distribution:
+---------------+-----+
|agreement_type |count|
+---------------+-----+
|employment     |10020|
|security       |7145 |
|purchase&ma    |4974 |
|services&supply|4037 |
|shareholder    |3147 |
|lease          |1744 |
+---------------+-----+



### CLASSICAL ML MODELS

**DEFINE LOGISTIC REGRESSION**

In [12]:
# --- Create a native Spark map for class weights
class_weight_map = create_map(
    [lit(float(x)) for x in chain.from_iterable(class_weights.items())]
)

# --- Add weight column to training dataframe
train_df = train_df.withColumn("class_weight", class_weight_map[col("label_indexed")])

# --- Define Logistic Regression with weights
lr = LogisticRegression(
    featuresCol="tfidf_features",
    labelCol="label_indexed",
    weightCol="class_weight",
    maxIter=180,
    regParam=0.01,
    elasticNetParam=0.1
)

**TRAIN MODEL-LOGISTIC REGRESSION**

In [13]:
# --- Train model
lr_model = lr.fit(train_df)

25/08/30 11:17:58 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB
25/08/30 11:18:00 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB
25/08/30 11:18:00 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/08/30 11:18:00 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB
25/08/30 11:18:02 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB
25/08/30 11:18:02 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB
25/08/30 11:18:03 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB
25/08/30 11:18:03 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB
25/08/30 11:18:04 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB
25/08/30 11:18:04 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB
25/08/30 11:18:05 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB
25/08/30 11:18:05 WARN DAGSchedul

**MODEL VALIDATION-LOGISTIC REGRESSION**

In [14]:
# --- Validate
val_predictions = lr_model.transform(val_df)

**MODEL EVALUATION-LOGISTIC REGRESSION**

In [15]:
# --- Evaluation
evaluator = MulticlassClassificationEvaluator(
    labelCol="label_indexed",
    predictionCol="prediction",
    metricName="f1"
)
val_f1 = evaluator.evaluate(val_predictions)
print(f"Validation F1: {val_f1:.4f}")

25/08/30 11:18:37 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB


Validation F1: 0.9002


**PERFORMANCE ANALYSIS-LOGISTIC REGRESSION**

In [16]:
# Get detailed metrics for the validation set
# Calculate multiple metrics
evaluators = {
    "f1": MulticlassClassificationEvaluator(labelCol="label_indexed", predictionCol="prediction", metricName="f1"),
    "accuracy": MulticlassClassificationEvaluator(labelCol="label_indexed", predictionCol="prediction", metricName="accuracy"),
    "weightedPrecision": MulticlassClassificationEvaluator(labelCol="label_indexed", predictionCol="prediction", metricName="weightedPrecision"),
    "weightedRecall": MulticlassClassificationEvaluator(labelCol="label_indexed", predictionCol="prediction", metricName="weightedRecall")
}

print("Logistic Regression Performance Metrics:")
for metric_name, evaluator in evaluators.items():
    score = evaluator.evaluate(val_predictions)
    print(f"{metric_name}: {score:.4f}")

Logistic Regression Performance Metrics:


25/08/30 11:18:39 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
25/08/30 11:18:40 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB


f1: 0.9002


25/08/30 11:18:42 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB


accuracy: 0.8984


25/08/30 11:18:43 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB


weightedPrecision: 0.9048


weightedRecall: 0.8984


In [17]:
for metric in ["f1", "precisionByLabel", "recallByLabel"]:
    evaluator = MulticlassClassificationEvaluator(
        labelCol="label_indexed",
        predictionCol="prediction",
        metricName=metric
    )
    print(metric, evaluator.evaluate(val_predictions))

25/08/30 11:18:45 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
25/08/30 11:18:46 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB


f1 0.9001809894875469


25/08/30 11:18:47 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB


precisionByLabel 0.935851966075559


recallByLabel 0.8727351164797239


**PERFROMANCE ANALYSIS BY CONTRACT TYPE-LOGISTIC REGRESSION**

In [18]:
labels = feature_df.select("label_indexed").distinct().orderBy("label_indexed").collect()
evaluator = MulticlassClassificationEvaluator(
    labelCol="label_indexed",
    predictionCol="prediction"
)
for row in labels:
    label = row["label_indexed"]
    r = evaluator.setMetricName("recallByLabel").evaluate(val_predictions, {evaluator.metricLabel: label})
    p = evaluator.setMetricName("precisionByLabel").evaluate(val_predictions, {evaluator.metricLabel: label})
    print(f"Label {label}: precision={p:.3f}, recall={r:.3f}")

25/08/30 11:18:49 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
25/08/30 11:18:51 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
25/08/30 11:18:52 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB


Label 0.0: precision=0.936, recall=0.873


25/08/30 11:18:53 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB


Label 1.0: precision=0.978, recall=0.930


25/08/30 11:18:55 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
25/08/30 11:18:56 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
25/08/30 11:18:58 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB


...

Label 3.0: precision=0.738, recall=0.893


25/08/30 11:19:01 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
25/08/30 11:19:02 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
25/08/30 11:19:04 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB


Label 4.0: precision=0.862, recall=0.865


25/08/30 11:19:05 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB


Label 5.0: precision=0.871, recall=0.899


**CONFUSION MATRIX TO IDENTIFY PROBLEM AREAS-LOGISTIC REGRESSION**

In [19]:
# Create confusion matrix
val_predictions.groupBy("label_indexed", "prediction").count().orderBy("label_indexed", "prediction").show()

# Show label mapping for interpretation
print("\nLabel Mapping:")
val_predictions.select("agreement_type", "label_indexed").distinct().orderBy("label_indexed").show()

25/08/30 11:19:06 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
25/08/30 11:19:08 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB


+-------------+----------+-----+
|label_indexed|prediction|count|
+-------------+----------+-----+
|          0.0|       0.0| 6069|
|          0.0|       1.0|   62|
|          0.0|       2.0|   29|
|          0.0|       3.0|  307|
|          0.0|       4.0|  355|
|          0.0|       5.0|  132|
|          1.0|       0.0|   90|
|          1.0|       1.0| 9220|
|          1.0|       2.0|   19|
|          1.0|       3.0|  407|
|          1.0|       4.0|   80|
|          1.0|       5.0|   97|
|          2.0|       0.0|   32|
|          2.0|       1.0|    9|
|          2.0|       2.0| 1579|
|          2.0|       3.0|   62|
|          2.0|       4.0|   18|
|          2.0|       5.0|    2|
|          3.0|       0.0|   99|
|          3.0|       1.0|   55|
+-------------+----------+-----+
only showing top 20 rows


Label Mapping:
+---------------+-------------+
| agreement_type|label_indexed|
+---------------+-------------+
|       security|          0.0|
|     employment|          1.0|
|     

**SparkNLP LegalBERT Classifier Pipeline**

In [22]:
# Document Assembler
document = DocumentAssembler() \
    .setInputCol("description") \
    .setOutputCol("document")

# Tokenizer
tokenizer = Tokenizer() \
    .setInputCols(["document"]) \
    .setOutputCol("token")

# Pretrained LegalBERT embeddings
embeddings = BertEmbeddings.pretrained("legal_bert_base_uncased", "en") \
    .setInputCols(["document", "token"]) \
    .setOutputCol("embeddings") \
    .setCaseSensitive(False)

# Sentence embeddings (pool word embeddings → sentence/document vector)
sentence_embeddings = SentenceEmbeddings() \
    .setInputCols(["document", "embeddings"]) \
    .setOutputCol("sentence_embeddings") \
    .setPoolingStrategy("AVERAGE")

# Neural classifier
classifier = ClassifierDLApproach() \
    .setInputCols(["sentence_embeddings"]) \
    .setOutputCol("class") \
    .setLabelColumn("type_label") \
    .setBatchSize(64) \
    .setMaxEpochs(20) \
    .setLr(1e-3) \
    .setEnableOutputLogs(True)

# Build SparkNLP pipeline
nlp_pipeline = Pipeline(stages=[
    document,
    tokenizer,
    embeddings,
    sentence_embeddings,
    classifier
])


legal_bert_base_uncased download started this may take some time.
Approximate size to download 388.4 MB
[ | ]

25/08/30 11:21:59 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
25/08/30 11:22:00 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


legal_bert_base_uncased download started this may take some time.
Approximate size to download 388.4 MB
[ / ]Download done! Loading the resource.
[ — ]Using CPUs
[OK!]


In [25]:
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import Tokenizer, BertEmbeddings
from pyspark.ml import Pipeline

# 1. Assemble document
document = DocumentAssembler() \
    .setInputCol("description") \
    .setOutputCol("document")

# 2. Tokenize
tokenizer = Tokenizer() \
    .setInputCols(["document"]) \
    .setOutputCol("token")

# 3. Load pretrained LegalBERT embeddings
embeddings = BertEmbeddings.pretrained("legal_bert_base_uncased", "en") \
    .setInputCols(["document", "token"]) \
    .setOutputCol("embeddings") \
    .setCaseSensitive(False)

# Build pipeline
pipeline = Pipeline(stages=[document, tokenizer, embeddings])


# Run pipeline
model = pipeline.fit(df)
result = model.transform(df)

# Show tokens + embeddings
result.selectExpr("token.result as tokens", "embeddings.embeddings as embeddings").show()


legal_bert_base_uncased download started this may take some time.


25/08/30 11:24:38 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


Approximate size to download 388.4 MB
[OK!]
+--------------------+--------------------+
|              tokens|          embeddings|
+--------------------+--------------------+
|[form, of, amende...|[[-0.1028678, -0....|
|[amendment, and, ...|[[-0.12935546, 0....|
|[exhibit, 99.2, c...|[[0.014824238, 0....|
|[exhibit, 10.44, ...|[[0.61817265, 0.3...|
|[securities, purc...|[[-0.372703, 0.68...|
|[waiver, concerni...|[[0.12642792, -0....|
|[amendment, #4, t...|[[-0.271368, -0.1...|
|[specimen, subscr...|[[-0.37500942, -0...|
|[form, of, securi...|[[-0.007023016, -...|
|[10.2, term, loan...|[[-0.34021887, 0....|
|[exhibit, 10.9, -...|[[0.4116112, 0.08...|
|[assignment, ,, a...|[[0.36096212, 0.0...|
|[amendment, numbe...|[[-0.6176207, -0....|
|[promis, note, ex...|[[-0.025361156, -...|
|[exhibit, a-2, -,...|[[0.5457407, -0.1...|
|[securities, purc...|[[-0.023123587, 0...|
|[convertible, pro...|[[0.34924334, 0.4...|
|[mortgage, securi...|[[-0.7191872, 0.5...|
|[senior, revolvin...|[[-1.10123

In [29]:
spark.stop()